In [1]:
import os
from huggingface_hub import snapshot_download
import scanpy as sc
import numpy as np

In [2]:
# input data
hepg2_raw_adata_path = "/data/storage/Replogle/raw_data/GSE264667_hepg2_raw_singlecell_01.h5ad"
preprocessed_adata_path = "/data/storage/Replogle/processed_data/replogle.h5ad"

# intermediate data
preprocessed_hepg2_path = "/data/storage/Replogle/processed_data/replogle_hepg2.h5ad"
processed_sub_hepg2_path = "/data/storage/Replogle/processed_data/replogle_hepg2_sub.h5ad"

# output data
raw_sub_hepg2_path = "/data/storage/Replogle/raw_data/GSE264667_hepg2_raw_singlecell_01_sub.h5ad"
preprocessed_hepg2_sub_ds_ntc_path = "/data/storage/Replogle/processed_data/replogle_hepg2_sub.h5ad"

In [9]:
## Read adata
# hepg2_raw_adata = sc.read_h5ad(hepg2_raw_adata_path)
raw_sub_ds_ntc_adata = sc.read_h5ad(raw_sub_hepg2_path)

# preprocessed_adata = sc.read_h5ad(preprocessed_adata_path)
# preprocessed_hepg2_adata = sc.read_h5ad(preprocessed_hepg2_path)
preprocessed_sub_ds_ntc_adata = sc.read_h5ad(preprocessed_hepg2_sub_ds_ntc_path)

In [6]:
genes_list=["MYC", "MYBL2", "CDK1", "TFAM", "MTOR",
            "STAT5B", "BAP1", "BRCA1", "SETD1A", "NELFA",
           "HAMP", "S100A1", "LRP5", "MAP2K7", "RHOQ"]
control=["non-targeting"]

# Calculate KD percentage

In [10]:
raw_token_map = dict(raw_sub_ds_ntc_adata.obs[["gene", "gene_id"]].drop_duplicates().values)

In [4]:
def knockdown(target_gene, adata, data, obs_col="gene_target", ntc="Non-Targeting"):
    if data =="raw":
        ensg_id = raw_token_map[target_gene]
        idx = adata.var_names.get_loc(ensg_id)
    else:
        idx = adata.var_names.get_loc(target_gene)

    
    gene_adata = adata[adata.obs[obs_col]==target_gene].copy()
    ntc_adata = adata[adata.obs[obs_col]==ntc].copy()

    gene_expression = gene_adata.X[:, idx]
    ntc_gene_expression = ntc_adata.X[:, idx]

    

    # pert = adata.obs[obs_col] == target_gene
    # ctrl = adata.obs[obs_col] == ntc
    # print(idx)
    # print(pert)

    pert_expr = gene_expression.mean()
    ctrl_expr = ntc_gene_expression.mean()

    log2FC = np.log2((pert_expr + 1e-6) / (ctrl_expr + 1e-6))
    knockdown = 1 - pert_expr / ctrl_expr
    
    print(target_gene)
    print(f"Mean target expression in perturbed: {pert_expr}")
    print(f"Mean target expression in control: {ctrl_expr}")
    print(f"log2FC: {log2FC}")
    print(f"knockdown: {knockdown}")
    print("\n")
    

    return {
        "gene": target_gene,
        "log2FC": log2FC,
        "knockdown": knockdown
    }

In [12]:
results = []
for gene in genes_list[:-1]:
    result_dict = knockdown(gene, preprocessed_sub_ds_ntc_adata, "preprocessed", obs_col="gene", ntc="non-targeting")
    results.append(result_dict)

results

MYC
Mean target expression in perturbed: 0.09259430319070816
Mean target expression in control: 0.6067663431167603
log2FC: -2.712132453918457
knockdown: 0.8473970890045166


MYBL2
Mean target expression in perturbed: 0.008471238426864147
Mean target expression in control: 0.5672177076339722
log2FC: -6.065018177032471
knockdown: 0.9850652813911438


CDK1
Mean target expression in perturbed: 0.01923966035246849
Mean target expression in control: 0.4846641421318054
log2FC: -4.654757976531982
knockdown: 0.9603031277656555


TFAM
Mean target expression in perturbed: 0.0402241051197052
Mean target expression in control: 0.6182818412780762
log2FC: -3.942099094390869
knockdown: 0.9349421262741089


MTOR
Mean target expression in perturbed: 0.0
Mean target expression in control: 0.30680009722709656
log2FC: -18.226943969726562
knockdown: 1.0




KeyError: 'STAT5B'

# Calculate pearson correlation

In [13]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy import sparse

In [14]:
def calc_pearson_correlation(gene_name, adata):
    # WARNING: do log-normalizing for raw counts
    target = gene_name
    target_col = "gene"           
    control_label = "non-targeting"  
    
    # select cells perturbed on given target genes and control cells
    target_cells = adata[adata.obs[target_col] == target].copy()
    nt_cells = adata[adata.obs[target_col] == control_label].copy()
    
    # randomly split target gene cells into two groups
    np.random.seed(1)
    idx = np.random.permutation(target_cells.n_obs)
    half = len(idx) // 2
    
    group1 = target_cells[idx[:half]]
    group2 = target_cells[idx[half:]]
    
    #  calculate mean expression
    def mean_expr(x):
        X = x.X
        if sparse.issparse(X):
            X = X.toarray()
        return np.asarray(X).mean(axis=0)
    
    mean_g1 = mean_expr(group1)
    mean_g2 = mean_expr(group2)
    mean_nt = mean_expr(nt_cells)
    
    # calculate gene expression vectors
    vec_g1 = mean_g1 - mean_nt
    vec_g2 = mean_g2 - mean_nt
    
    # calculate Pearson correlation
    r, p = pearsonr(vec_g1, vec_g2)
    
    print("Pearson r:", r)
    print("p-value:", p)
    return r, p

In [15]:
calc_pearson_correlation("MYC", preprocessed_sub_ds_ntc_adata)

Pearson r: 0.6151137
p-value: 0.0


(np.float32(0.6151137), np.float32(0.0))

# sgRNA abundance

In [16]:
sgrna_df = pd.read_csv("/data/storage/X-Atlas/HCT116/h5ad/HCT116_filtered_guide_calls_per_cell.csv")
sgrna_df

,cell_barcode,num_features,feature_call,num_umis,gene_target,pass_guide_filter
0,AAACCAAAGACATGTT-HCT116_Batch1,2,ST14_P1P2-1|ST14_P1P2-2,1794|1481,ST14,True
1,AAACCAAAGACCCAAC-HCT116_Batch1,2,SIGLEC5_P1P2-1|SIGLEC5_P1P2-2,1429|962,SIGLEC5,True
2,AAACCAAAGAGGTACG-HCT116_Batch1,2,VSNL1_P1P2-1|VSNL1_P1P2-2,1284|2990,VSNL1,True
3,AAACCAAAGCGATTAT-HCT116_Batch1,2,KCNK7_P1P2-1|KCNK7_P1P2-2,1626|1382,KCNK7,True
4,AAACCAAAGGCTTAAT-HCT116_Batch1,2,APOA4_P1P2-1|APOA4_P1P2-2,5347|2860,APOA4,True
...,...,...,...,...,...,...
3409164,GTTGTGCAGGGAATCG-HCT116_Batch109,2,ACP6_P1P2-1|ACP6_P1P2-2,130|705,ACP6,True
3409165,GTTGTGGGTCACGCGG-HCT116_Batch109,2,RNF39_P1P2-1|RNF39_P1P2-2,86|58,RNF39,True
3409166,GTTGTGGGTCTGACTC-HCT116_Batch109,2,AFP_P2-1|AFP_P2-2,22|41,AFP,True
3409167,GTTGTGGGTCTTGATT-HCT116_Batch109,2,METTL9_P1P2-1|METTL9_P1P2-2,558|356,METTL9,True
